
- The GPU server runs a **shared JupyterHub environment** (`/opt/conda`) that all users share, meaning any admin update affects everyone's work without warning.

- On **June 9th**, the server's `torch` was upgraded from the previously working version to `torch 2.12.0+cu130`, but `torchvision` was not updated to match, causing a C++ extension mismatch (`torchvision::nms does not exist`).

- The new `torch 2.12.0+cu130` also requires **CUDA 13.0**, but the server's NVIDIA driver only supports **CUDA 12.4**, making the admin-installed torch version entirely unusable.

- The workaround is to install a compatible stack (`torch==2.6.0`, `torchvision==0.21.0`, cu124) into the **user directory** (`~/.local`) each session, since there is no personal conda environment available.

- `torchaudio` caused an additional conflict — even when not explicitly installed, a leftover version in `~/.local` from a previous session was being picked up by `transformers`, crashing on import due to the CUDA 13 runtime mismatch. It must be explicitly uninstalled each session.

- `transformers` must be installed **from source** (GitHub) each session because Qwen3-VL support is not yet in a stable PyPI release.

- Because `~/.local` is also wiped on server restarts, the **entire stack must be reinstalled at the start of every session**, adding overhead to each working day.

- The root cause of all issues is the **lack of a personal conda environment**, which would isolate the dependency stack from server-wide changes and eliminate the need for per-sesease.sion reinstalls.

In [ ]:
# This installs torch 2.6.0 and torchvision 0.21.0 pinned to cu124 because the
# GPU server's NVIDIA driver supports CUDA 12.4 but not CUDA 13.0. torchaudio is
# explicitly uninstalled and excluded because it is not needed for Qwen3-VL and
# its cu124 build conflicts with the server's CUDA 13 runtime libraries, causing
# an OSError on import. transformers is installed from source because Qwen3-VL
# support is not yet available in a stable PyPI release.

import sys, subprocess

# Uninstall torchaudio if it exists from a previous session
subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

# Install torch stack (no torchaudio)
subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

# Install transformers + dependencies
subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate", "qwen-vl-utils",
    "--user", "-q"], check=True)

print("✅ Done — restart the kernel now")

In [6]:
!nvidia-smi

Wed Jun 10 01:30:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:CF:00.0 Off |                   On |
| N/A   29C    P0             77W /  700W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [2]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")

model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("✅ Loaded successfully")

Using device: cuda


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

✅ Loaded successfully


In [ ]:
"""
# old code
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct",
    trust_remote_code=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct",
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print("✅ Loaded successfully")
"""